# 02 — Grounding and Variables

**Source:** `tutorials/1-grounding_non_logical_symbols.ipynb`

In notebook 01 we used `ltn.Constant`, `ltn.Variable`, `ltn.Predicate.MLP` without looking closely at what they produce as tensors.
This notebook fixes that: understanding the shapes is essential before you can write your own axioms.

**What you will see:**
- The four kinds of LTN symbols: constants, variables, predicates, functions
- Two ways to define a predicate: Lambda vs custom Keras model
- What `ltn.Function` is (and how it differs from a predicate)
- **Variable broadcasting**: why `P(x, y)` with |x|=10, |y|=5 produces a (10,5) tensor
- The `free_vars` attribute that tracks which tensor axis belongs to which variable
- How quantifiers collapse those axes (connecting back to the fold in notebook 01)

## 0. Imports

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tensorflow as tf
import ltn

## 1. Constants

A constant is a single individual, a fixed point in some tensor space.
It can be any rank: a scalar, a vector, a matrix, …

`trainable=False` → fixed data point (e.g. a sensor reading)
`trainable=True`  → learnable embedding (what we used for people in the previous notebook 01)

In [2]:
# Non-trainable: fixed feature vectors
c1 = ltn.Constant([2.1, 3.0], trainable=False)
c2 = ltn.Constant([[4.2, 3.0, 2.5],
                    [4.0,-1.3, 1.8]], trainable=False)   # rank-2 constant

print("c1:", c1)
print("c1.tensor:", c1.tensor)
print("c1.free_vars:", c1.free_vars)   # empty, a constant has no free variables
print()
print("c2 shape:", c2.tensor.shape)    # (2,3) matrix

c1: ltn.Constant(tensor=[2.1 3. ], trainable=False, free_vars=[])
c1.tensor: tf.Tensor([2.1 3. ], shape=(2,), dtype=float32)
c1.free_vars: []

c2 shape: (2, 3)


In [3]:
# Trainable: the embedding is a tf.Variable, gradients will update it
c3 = ltn.Constant([0., 0.], trainable=True)
print("c3:", c3)
print("c3.tensor type:", type(c3.tensor))   # tf.Variable, not tf.Tensor

c3: ltn.Constant(tensor=<tf.Variable 'Variable:0' shape=(2,) dtype=float32, numpy=array([0., 0.], dtype=float32)>, trainable=True, free_vars=[])
c3.tensor type: <class 'tensorflow.python.ops.resource_variable_ops.ResourceVariable'>


---
### ❓ Q — What is `free_vars` and why is it empty for a constant?

`free_vars` is a list of variable **labels** that are still "open" in an expression:
axes of the tensor that correspond to a variable that has not yet been quantified away.

A constant has no free variables: it represents one specific individual, so its tensor is just
a fixed array with no more "for each" dimensions. `free_vars = []`.

A variable introduces a free axis. A quantifier (`Forall`, `Exists`) closes it.
We will see this progression clearly in section 4.

## 2. Predicates

A predicate maps one or more individuals to a truth value in [0,1].

**Two ways to define one:**

| Constructor | Use when |
|-------------|----------|
| `ltn.Predicate.Lambda(fn)` | Simple closed-form formula, no trainable weights |
| `ltn.Predicate(keras_model)` | Anything learnable (MLP, CNN, …) |

`ltn.Predicate.MLP(...)` from notebook 01 is just a convenience wrapper around the second option.

In [15]:
# --- Option 1: Lambda predicate (non-trainable) ---
# P1(x) = exp(-‖x - μ‖²)  —> measures proximity to a fixed point μ
mu = tf.constant([2., 3.])
P1 = ltn.Predicate.Lambda(lambda x: tf.exp(-tf.norm(x - mu, axis=1)))

# --- Option 2: custom Keras model (trainable) ---
class ModelP2(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(5, activation=tf.nn.elu)
        self.dense2 = tf.keras.layers.Dense(1, activation=tf.nn.sigmoid)  # output in [0,1]

    def call(self, x):
        return self.dense2(self.dense1(x))

P2 = ltn.Predicate(ModelP2())

In [5]:
# Evaluate on a constant, result is a Formula with no free vars (a single scalar)
c1 = ltn.Constant([2.1, 3.0], trainable=False)
c2 = ltn.Constant([4.5, 0.8], trainable=False)

print("P1(c1):", P1(c1))          # close to mu=(2,3) → should be high
print("P1(c2):", P1(c2))          # far from mu → should be lower
print("P2(c1):", P2(c1))          # random initialisation

P1(c1): ltn.Formula(tensor=0.9048374891281128, free_vars=[])
P1(c2): ltn.Formula(tensor=0.0357871949672699, free_vars=[])
P2(c1): ltn.Formula(tensor=0.38360336422920227, free_vars=[])


---
### ❓ Q — When would I use a Lambda predicate vs an MLP?

**Lambda predicate:** when the truth function is known and fixed.
Examples: `IsSimilar(x,y) = exp(-‖x-y‖)`, `IsPositive(x) = sigmoid(x[0])`,
a hard constraint like `SameClass(x,y) = 1 if label(x)==label(y) else 0`.
No weights — no learning.

**MLP (or custom Keras model):** when you want the predicate to be *learned from data*.
The MLP weights are the parameters; training adjusts them so the predicate agrees with
the axioms. This is the standard case in LTN applications.

You can also mix: a non-trainable predicate can encode hard constraints (physics laws,
symmetry rules) while trainable predicates learn from data. Both appear in the same KB.

### Multi-input predicates

When a predicate takes more than one individual, pass them as a **list**.
The Keras model receives a list of tensors and must concatenate them itself (axis=1, not axis=0,
because axis=0 is the batch dimension).

In [6]:
class ModelP3(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(5, activation=tf.nn.elu)
        self.dense2 = tf.keras.layers.Dense(1, activation=tf.nn.sigmoid)

    def call(self, inputs):
        x1, x2 = inputs[0], inputs[1]
        x = tf.concat([x1, x2], axis=1)   # axis=1: feature dim, not batch dim
        return self.dense2(self.dense1(x))

P3 = ltn.Predicate(ModelP3())
print("P3([c1,c2]):", P3([c1, c2]))   # two constants → one scalar

P3([c1,c2]): ltn.Formula(tensor=0.8316042423248291, free_vars=[])


## 3. Functions

A function maps individuals to **another individual** (a term, not a truth value).
Output is a tensor in the domain, not in [0,1].

Use functions when you need to compute intermediate representations:
- difference between two embeddings
- projection into a lower-dimensional space
- centroid of a set of points

Same two constructors as predicates: `ltn.Function(model)` or `ltn.Function.Lambda(fn)`.

In [7]:
# f1: vector difference (non-trainable)
f1 = ltn.Function.Lambda(lambda args: args[0] - args[1])

# f2: learned projection to R^5
class MyModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(4, activation=tf.nn.relu)
        self.dense2 = tf.keras.layers.Dense(5)  # no sigmoid — output is a vector, not [0,1]

    def call(self, x):
        return self.dense2(self.dense1(x))

f2 = ltn.Function(MyModel())

print("f1([c1,c2]):", f1([c1, c2]))   # returns a Term (vector), not a Formula
print("f2(c1):",      f2(c1))          # projects c1 to R^5

f1([c1,c2]): ltn.Term(tensor=[-2.4  2.2], free_vars=[])
f2(c1): ltn.Term(tensor=[ 0.5818568  -0.73645645 -0.16185269 -0.12835236 -0.10994022], free_vars=[])


---
### ❓ Q — What is the difference between a Function and a Predicate?

| | Predicate | Function |
|-|-----------|----------|
| Output type | `ltn.Formula` — truth value in [0,1] | `ltn.Term` — a tensor in the domain |
| Typical use | Classification, similarity, constraints | Feature transformation, centroid, difference |
| Can be composed | Inside connectives/quantifiers | As input to predicates or other functions |

Example of composing them:
```python
# IsClose(x, y) = P1(f1([x, y]))
# f1 computes the difference vector, P1 measures how small it is
IsClose = ltn.Predicate.Lambda(
    lambda diff: tf.exp(-tf.norm(diff, axis=1))
)
# then: IsClose(f1([x, y]))
```

Functions let you build richer predicates without hardcoding the arithmetic inside the predicate model.

## 4. Variables and broadcasting  ← the key section

A variable is a **sequence of individuals** — essentially a batch of constants stacked together.
Variables are used to write quantified statements: `∀x P(x)`, `∃y Friends(x,y)`.

When you evaluate a predicate on a variable, LTN evaluates it on **every individual at once**
and returns a tensor with one axis per variable.

In [8]:
x = ltn.Variable('x', np.random.normal(0., 1., (10, 2)))   # 10 individuals in R^2
y = ltn.Variable('y', np.random.normal(0., 4., ( 5, 2)))   #  5 individuals in R^2

print("x.tensor.shape:", x.tensor.shape)   # (10, 2)
print("x.free_vars:   ", x.free_vars)      # ['x']
print()
print("y.tensor.shape:", y.tensor.shape)   # (5, 2)
print("y.free_vars:   ", y.free_vars)      # ['y']

x.tensor.shape: (10, 2)
x.free_vars:    ['x']

y.tensor.shape: (5, 2)
y.free_vars:    ['y']


In [9]:
# P2 on a single variable → one output per individual
res_x = P2(x)
print("P2(x).tensor.shape:", res_x.tensor.shape)   # (10,)
print("P2(x).free_vars:   ", res_x.free_vars)       # ['x']   one free axis left

P2(x).tensor.shape: (10,)
P2(x).free_vars:    ['x']


In [10]:
# P3 on TWO variables → one output per (individual_x, individual_y) PAIR
# This is broadcasting: LTN computes the full outer product of individuals
res_xy = P3([x, y])
print("P3([x,y]).tensor.shape:", res_xy.tensor.shape)   # (10, 5)
print("P3([x,y]).free_vars:   ", res_xy.free_vars)       # ['x', 'y']
print()
print("Interpretation: cell [i,j] = P3(x_i, y_j)")
print("Row 0 (x_0 paired with every y):", res_xy.tensor[0].numpy().round(3))

P3([x,y]).tensor.shape: (10, 5)
P3([x,y]).free_vars:    ['x', 'y']

Interpretation: cell [i,j] = P3(x_i, y_j)
Row 0 (x_0 paired with every y): [0.001 0.845 0.85  0.841 0.34 ]


---
### ❓ Q — Why does P3([x,y]) produce a (10,5) tensor? I expected (10,) or (50,).

This is the core of LTN's design and the thing that confuses people most.

In classical FOL, `∀x ∀y P(x,y)` quantifies over **all pairs** (x,y).
To implement this efficiently, LTN evaluates P on the **cartesian product** of x and y at once —
that is exactly the (10,5) tensor.

- Axis 0 (size 10) = the x variable
- Axis 1 (size 5)  = the y variable
- Cell [i,j] = truth value of P(xᵢ, yⱼ)

Then `Forall([x,y], P([x,y]))` **folds both axes** into a single scalar.

**Partial quantification — which axis survives?**
The axis of the variable you quantify is the one that disappears.
The remaining variable's axis is what you get back.

| Expression | Folds | Result shape | Surviving axis | Meaning of each output cell |
|------------|-------|-------------|----------------|-----------------------------|
| `Forall([x,y], P([x,y]))` | both | `()` | — | universal over all pairs |
| `Forall(x, P([x,y]))` | x (axis 0) | `(5,)` | y | for each yⱼ: how universally does P hold across all x? |
| `Forall(y, P([x,y]))` | y (axis 1) | `(10,)` | x | for each xᵢ: how universally does P hold across all y? |

So `Forall(x, ...)` removes the **x** axis and leaves **y** ,  not the other way around.
A memory aid: the variable you name in `Forall` is the one that gets folded away.

**(50,)** would mean you flattened the pairs into a list — that loses the structure.
LTN keeps the axes separate precisely so partial quantification works correctly.

In [11]:
# Connecting to notebook 01: Forall folds the free variable axes
Forall = ltn.Wrapper_Quantifier(ltn.fuzzy_ops.Aggreg_pMeanError(p=2), semantics="forall")

# Fold both axes → scalar
both_folded = Forall([x, y], P3([x, y]))
print("Forall([x,y], P3([x,y])) shape:", both_folded.tensor.shape)   # ()
print("Forall([x,y], P3([x,y])) value:", both_folded.tensor.numpy().round(4))
print()

# Fold only x → one value per y individual
x_folded = Forall(x, P3([x, y]))
print("Forall(x, P3([x,y])) shape:", x_folded.tensor.shape)   # (5,)
print("Forall(x, P3([x,y])) values:", x_folded.tensor.numpy().round(3))
print("(for each y_j: how universally true is P across all x?)")

Forall([x,y], P3([x,y])) shape: ()
Forall([x,y], P3([x,y])) value: 0.4629

Forall(x, P3([x,y])) shape: (5,)
Forall(x, P3([x,y])) values: [0.001 0.847 0.851 0.844 0.387]
(for each y_j: how universally true is P across all x?)


---
### ❓ Q — What happens when I mix a constant and a variable?

A constant has no free axis, it is broadcast against the variable's axis.

In [12]:
# c1 is a constant (no free axis), y is a variable (5 individuals)
res_cy = P3([c1, y])
print("P3([c1,y]).tensor.shape:", res_cy.tensor.shape)   # (5,)
print("P3([c1,y]).free_vars:   ", res_cy.free_vars)       # ['y'] only
print()
print("Interpretation: cell [j] = P3(c1, y_j) — c1 is fixed, y varies")
print("Values:", res_cy.tensor.numpy().round(3))

P3([c1,y]).tensor.shape: (5,)
P3([c1,y]).free_vars:    ['y']

Interpretation: cell [j] = P3(c1, y_j) — c1 is fixed, y varies
Values: [0.018 0.845 0.849 0.84  0.535]


## 5. Indexing into a result with `take`

`result.take(var_label, index)` slices along the axis of a specific variable.
Useful for inspecting individual pairs.

In [13]:
res_xy = P3([x, y])

# Get the truth value for the 3rd individual in x paired with the 1st individual in y
cell = res_xy.take('x', 2).take('y', 0)
print("P3(x[2], y[0]):", cell.tensor.numpy().round(4))

# Verify manually
manual = P3([x.take('x', 2), y.take('y', 0)])
print("Manual P3(x[2], y[0]):", manual.tensor.numpy().round(4))

P3(x[2], y[0]): 0.0018
Manual P3(x[2], y[0]): 0.0018


## 6. Variables from trainable constants

In notebook 01 we used `ltn.Variable.from_constants(...)` inside the `axioms` function.
To understand why, we first need to understand what a `GradientTape` is.

---
### ❓ Q — What is a GradientTape?

Training a neural network means computing **how much each parameter should change**
to reduce the loss. That requires computing gradients, derivatives of the loss
with respect to every trainable parameter.

TensorFlow does this with `tf.GradientTape`: a recorder that **watches** all operations
performed inside its `with` block and builds a computation graph as it goes.
When you call `tape.gradient(loss, variables)` at the end, it walks that graph
backwards (backpropagation) and returns the gradient for each variable.

```
with tf.GradientTape() as tape:
    output = model(input)       # tape records this operation
    loss   = 1 - output

grads = tape.gradient(loss, model.trainable_variables)
# grads[i] = d(loss)/d(variable[i])
optimizer.apply_gradients(zip(grads, model.trainable_variables))
```

Without the tape, TensorFlow does not know which operations to differentiate,
there is nothing to walk back through.

---
### ❓ Q — Why does `Variable.from_constants` need to be inside the tape?

When constants are `trainable=True`, their embedding vectors are `tf.Variable` objects.
`Variable.from_constants` stacks those vectors into a single tensor (the variable's batch).

The tape needs to **see that stacking operation** happen inside its scope so it can
track the connection: `loss → predicate output → variable tensor → constant embeddings`.
If you build the variable outside the tape, the tape misses the link between the variable
and the constants, and `tape.gradient(..., c1.tensor)` returns `None`.

This is also why `axioms()` in notebook 01 rebuilds `p` and `q` at every training step:
the optimizer updates the constant embeddings each step, so the variable must be
reconstructed inside a fresh tape to reflect the new values and maintain gradient tracking.

In [16]:
c1 = ltn.Constant([2.1, 3.0], trainable=True)
c2 = ltn.Constant([4.5, 0.8], trainable=True)

with tf.GradientTape() as tape:
    # The tape records operations on c1, c2 so gradients flow back to them
    x_var = ltn.Variable.from_constants("x", [c1, c2], tape=tape)
    res = P2(x_var)

# Gradient of the output w.r.t. c1's embedding vector
grad = tape.gradient(res.tensor, c1.tensor)
print("d(P2(x)) / d(c1):", grad.numpy())
print()
print("This is why axioms() rebuilds variables each step:")
print("the constants are updated by the optimizer, so the variable")
print("must be reconstructed to reflect the new embedding values.")

d(P2(x)) / d(c1): [0.00347377 0.00491301]

This is why axioms() rebuilds variables each step:
the constants are updated by the optimizer, so the variable
must be reconstructed to reflect the new embedding values.


## Summary

| Symbol | LTN type | Tensor shape | `free_vars` |
|--------|----------|-------------|-------------|
| `ltn.Constant(v)` | `Constant` / `Term` | shape of v | `[]` |
| `ltn.Variable('x', data)` | `Variable` / `Term` | (n, features) | `['x']` |
| `P(constant)` | `Formula` | `()` scalar | `[]` |
| `P(variable_x)` | `Formula` | `(n,)` | `['x']` |
| `P([variable_x, variable_y])` | `Formula` | `(n, m)` | `['x','y']` |
| `f(variable_x)` | `Term` | `(n, out_features)` | `['x']` |
| `Forall([x,y], P([x,y]))` | `Formula` | `()` scalar | `[]` |
| `Forall(x, P([x,y]))` | `Formula` | `(m,)` | `['y']` |

**The key rule:** every variable introduced in an expression adds one axis to the output tensor.
Quantifiers fold (reduce) those axes away. A closed formula has shape `()` and `free_vars=[]`.

**Next:** `03_kb_and_learning.ipynb`, build a knowledge base with multiple axioms and train it (Tutorial 3 in the repo).